# Tool Use — What Does the Response Actually Look Like?

This notebook shows the raw API response for two calls:
1. A plain `messages.create` call — free text back
2. The same call with `tools` + `tool_choice` — structured JSON back

Same question. Same Claude. Different response shape.

In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# A simple test prompt — nothing to do with Pattern Mirror yet
# We'll use a tiny example so the response is easy to read
TEST_PROMPT = "Analyse this person: they always say yes to requests even when overloaded, and feel resentful afterwards."

## Call 1 — Plain text response
No tools. Claude responds however it wants.

In [2]:
plain_response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": TEST_PROMPT}]
)

print("--- FULL RESPONSE OBJECT ---")
print(plain_response)
print()
print("--- response.content ---")
print(plain_response.content)
print()
print("--- content[0].type ---")
print(plain_response.content[0].type)
print()
print("--- content[0].text ---")
print(plain_response.content[0].text)

--- FULL RESPONSE OBJECT ---
Message(id='msg_01QYbdHpHdzPViBeyM8YetPi', container=None, content=[TextBlock(citations=None, text='# Key patterns to examine\n\n**The immediate cycle:**\n- Says yes → accumulates obligations → feels resentment → but repeats it\n- This suggests the *cost of saying no* feels higher than the cost of resentment\n\n**Likely drivers:**\n- Fear of rejection or being seen as unhelpful/selfish\n- Difficulty distinguishing between being needed and being valued\n- Possible guilt about having limits\n- Discomfort with others\' disappointment (even brief, manageable disappointment)\n\n**What makes it stick:**\n- Resentment is a symptom, not the real problem—it\'s evidence the boundary was needed, but it doesn\'t change the underlying fear\n- Each "yes I did it anyway" reinforces: "See, I can handle it," which validates the next overcommitment\n\n**Worth exploring:**\n- Is the resentment directed at others, or at themselves? (Often both—anger at others for asking, plus 

## Call 2 — Tool use response

Now we define a schema and force Claude to call it.

Watch what changes:
- `content[0].type` → was `text`, now `tool_use`
- `content[0].text` → gone
- `content[0].input` → a Python dict matching our schema exactly

In [3]:
# Define a simple schema — just enough to show the shape
SIMPLE_TOOL = {
    "name": "pattern_analysis",
    "description": "Return a structured pattern analysis.",
    "input_schema": {
        "type": "object",
        "properties": {
            "pattern_name": {
                "type": "string",
                "description": "Short plain-language name for the pattern."
            },
            "what_it_protects_from": {
                "type": "string",
                "description": "What this pattern is protecting the person from."
            },
            "evidence": {
                "type": "array",
                "items": {"type": "string"},
                "description": "1-2 phrases from the description that reveal the pattern."
            }
        },
        "required": ["pattern_name", "what_it_protects_from", "evidence"]
    }
}

tool_response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    tools=[SIMPLE_TOOL],
    tool_choice={"type": "tool", "name": "pattern_analysis"},
    messages=[{"role": "user", "content": TEST_PROMPT}]
)

print("--- FULL RESPONSE OBJECT ---")
print(tool_response)
print()
print("--- response.content ---")
print(tool_response.content)
print()
print("--- content[0].type ---")
print(tool_response.content[0].type)
print()
print("--- content[0].input ---")
print(tool_response.content[0].input)

--- FULL RESPONSE OBJECT ---
Message(id='msg_01HftGjdkrCp8BuDwY6dtgiE', container=None, content=[ToolUseBlock(id='toolu_01BLVwW6y85cGvc845Fgi2A7', caller=DirectCaller(type='direct'), input={'pattern_name': 'Chronic Yes-Saying & Resentment', 'what_it_protects_from': 'Fear of rejection, conflict, or disappointing others; anxiety about being seen as selfish or unhelpful', 'evidence': ['always say yes to requests even when overloaded', 'feel resentful afterwards']}, name='pattern_analysis', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=774, output_tokens=125, server_tool_use=None, service_tier='standard'))

--- response.content ---
[ToolUseBlock(id='toolu_01BLVwW6y85cGvc845Fgi2A7', caller

## Accessing fields directly

Because `.input` is a Python dict, you can address any field by key.
No string parsing. No regex. No format assumptions.

In [4]:
data = tool_response.content[0].input

print("Pattern name:        ", data["pattern_name"])
print("Protects from:       ", data["what_it_protects_from"])
print("Evidence:")
for quote in data["evidence"]:
    print(f'  - "{quote}"')

Pattern name:         Chronic Yes-Saying & Resentment
Protects from:        Fear of rejection, conflict, or disappointing others; anxiety about being seen as selfish or unhelpful
Evidence:
  - "always say yes to requests even when overloaded"
  - "feel resentful afterwards"


## What `stop_reason` tells you

One more thing worth seeing — the `stop_reason` field on the response changes too.

In [5]:
print("Plain response stop_reason: ", plain_response.stop_reason)
print("Tool  response stop_reason: ", tool_response.stop_reason)

# Plain = 'end_turn'  → Claude decided it was done
# Tool  = 'tool_use' → Claude stopped because it made a tool call

Plain response stop_reason:  max_tokens
Tool  response stop_reason:  tool_use


## Now with the real Pattern Mirror schema

Same pattern — but now using the actual `ANALYSIS_TOOL` from `prompts.py`
and a mock reflection session so you can see the full output shape.

In [ ]:
import sys
sys.path.insert(0, ".")

from prompts import SYSTEM_PROMPT, ANALYSIS_TOOL

MOCK_ANSWERS = """<reflection_session>
  <answer id="Q1">I told myself I just wasn't organised enough, that I always leave things too late.</answer>
  <answer id="Q2">My colleague interrupted me mid-sentence in a meeting. I thought: they never take me seriously.</answer>
  <answer id="Q3">That I don't really know what I'm doing. I over-prepare so nobody notices. I'm worried they'd stop trusting me.</answer>
  <answer id="Q4">Feeling like I'm behind. It happened yesterday when I saw my inbox.</answer>
  <answer id="Q5">I'm just not someone who asks for help.</answer>
  <answer id="Q6">Yes — at home too. I take on everything myself and then feel invisible when no one notices.</answer>
</reflection_session>"""

full_response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    system=SYSTEM_PROMPT,
    tools=[ANALYSIS_TOOL],
    tool_choice={"type": "tool", "name": "pattern_analysis"},
    messages=[{"role": "user", "content": MOCK_ANSWERS}]
)

result = full_response.content[0].input
print("stop_reason:", full_response.stop_reason)
print()
print("Full dict:")
import json
print(json.dumps(result, indent=2))

In [ ]:
# Now access individual sections exactly as analyser.py does
print("Core pattern name:  ", result["core_pattern"]["name"])
print("Payoff:             ", result["payoff"])
print("Protocol trigger:   ", result["protocol"]["detection_trigger"])
print("Protocol steps:")
for i, step in enumerate(result["protocol"]["steps"], 1):
    print(f"  {i}. {step}")